# STPPY Quick Start

End-to-end walkthrough of the stppy analysis pipeline.
Follows the steps in `README.md`.

**Prerequisites:** preprocessed EEG data (artifact removal, ICA, bandpass filtering already done).

In [1]:
import os
import glob
import numpy as np
import mne
import stppy

## 1. Load and Preprocess EEG Data

In [ ]:
# Find all EEG files (recursively searches directories)
files = stppy.io.find_eeg_files('/home/novicky/psiconnect')

# Create the positions file (use a file with all electrodes present)
raw_complete = stppy.io.load_eeg_raw(files[7], preload=True)
stppy.io.create_electrode_positions_file(raw_complete, '/home/novicky/stppy/electrode_positions.json', overwrite=True)

In [ ]:
# Process all files
datasets = []
for i, file_path in enumerate(files):
    print(f"Processing {i+1}/{len(files)}: {file_path}")

    raw = stppy.io.load_eeg_raw(file_path, preload=True)

    raw_interp, interp_channels = stppy.preprocessing.interpolate_missing_electrodes(
        raw,
        positions_file='./electrode_positions.json',
        apply_average_reference=True,
        verbose=False
    )

    raw_250 = stppy.preprocessing.resample_data(raw_interp, 250)

    datasets.append(raw_250.get_data())

## 2. Detect Avalanches

In [ ]:
crossover_thresholds = [-3, 3]
filter_masks = stppy.detection.create_avalanche_filter(datasets, crossover_thresholds)

all_avalanches, all_avalanche_times = stppy.detection.detect_avalanches(
    datasets,
    filter_masks,
    crossover_thresholds,
    min_duration=11
)

In [ ]:
# Apply spatial filtering
K, channel_names = stppy.utils.create_eeg_ground_metric('./electrode_positions.json')

filtered_avalanches, filtered_times = stppy.detection.filter_avalanche_propagation(
    all_avalanches,
    all_avalanche_times,
    K,
    './electrode_positions.json',
    min_active_neighbors=5
)

# Convert to peak representation
peak_avalanches = stppy.detection.convert_to_peaks(filtered_avalanches)
stppy.io.save_pickle(peak_avalanches, './peak_avalanches.pkl')
stppy.io.save_pickle(filtered_times,  './peak_avalanche_times.pkl')

## 3. Compute Similarity Matrix

### Option A — Serial (< 5 000 avalanches)

In [ ]:
avalanches, avalanche_times = stppy.io.load_avalanche_data(
    './peak_avalanches.pkl',
    min_time_length=11,
    include_flipped=True,
    times_filepath='./peak_avalanche_times.pkl'
)

K, channel_names = stppy.utils.create_eeg_ground_metric('./electrode_positions.json')
K_norm = stppy.utils.normalize_distance_matrix(K, method='mean')

similarity_matrix, path_matrix = stppy.alignment.compute_full_similarity_matrix(
    avalanches,
    K_norm,
    show_progress=True,
    save_interval=100,
    checkpoint_file='./similarity_checkpoint.pkl'
)

stppy.io.save_pickle(similarity_matrix, './similarity_matrix.pkl')
stppy.io.save_pickle(path_matrix,       './path_matrix.pkl')

### Option B — Block-based (> 5 000 avalanches)

In [ ]:
# Step 1: Generate block plan
plan = stppy.alignment.generate_block_plan(
    n_avalanches=len(avalanches),
    block_size=2554
)
print(f"Need to compute {len(plan)} blocks")

In [ ]:
# Step 2: Each block is a standalone script in paper_scripts/blocks/.
# Run them in parallel on different machines/cores, e.g.:
#   python paper_scripts/blocks/block000.py
#   python paper_scripts/blocks/block001.py
#   ...
#   python paper_scripts/blocks/block009.py

In [ ]:
# Step 3: Merge and reconstruct
block_files = sorted(glob.glob('./blocks/block_*.pkl'))
merged = stppy.alignment.merge_blocks(block_files, output_file='./similarity_merged.pkl')

In [ ]:

similarity_matrix, path_matrix = stppy.alignment.reconstruct_full_matrices(
    merged,
    n_avalanches=len(avalanches)
)

stppy.io.save_pickle(similarity_matrix, './similarity_matrix.pkl')
stppy.io.save_pickle(path_matrix,       './path_matrix.pkl')

## 4. Cluster Avalanches

In [ ]:
# similarity_matrix = stppy.io.load_pickle('./similarity_matrix.pkl')
# path_matrix       = stppy.io.load_pickle('./path_matrix.pkl')

clusters, cluster_info = stppy.clustering.simple_hierarchical_clustering(
    similarity_matrix,
    threshold=3.5,
    method='average',
    min_cluster_size=30
)

print(f"{len(cluster_info)} clusters, {(clusters == -1).sum()} unassigned")

In [ ]:
# Build consensus alignment for each cluster
cluster_averages = {}
for cluster_id in range(len(cluster_info)):
    aligned, average, info, medoid_idx, counts = stppy.clustering.build_consensus_alignment(
        cluster_id=cluster_id,
        clusters=clusters,
        avalanche_arrays=avalanches,
        avalanche_paths=path_matrix,
        similarity_clean=similarity_matrix
    )
    cluster_averages[cluster_id] = average

# Remove polarity-flipped duplicates
clusters_to_keep = stppy.clustering.remove_duplicate_polarities(
    cluster_averages,
    threshold=-0.999
)
print(f"Clusters to keep: {clusters_to_keep}")

## 5. Detect Cluster Sequences in Continuous Data

In [ ]:
cluster_activation_times = {}
cluster_durations = {}

for cluster_id in clusters_to_keep:
    cluster_template = cluster_averages[cluster_id]
    cluster_durations[cluster_id] = cluster_template.shape[1] / 250.0

    activations = []
    for dataset_idx, continuous_data in enumerate(datasets):
        result = stppy.detection.detect_sequences_by_correlation(
            continuous_data,
            cluster_template,
            sfreq=250.0,
        )
        activations.append((dataset_idx, result['times'],
                            result['polarities'], result['correlations']))

    cluster_activation_times[cluster_id] = activations

stppy.io.save_pickle(
    {'cluster_activation_times': cluster_activation_times},
    './cluster_analysis.pkl'
)
stppy.io.save_pickle(cluster_durations, './cluster_durations.pkl')

In [ ]:
sequences_by_cluster = stppy.detection.detect_sequences_in_clusters(
    cluster_activation_times,
    cluster_durations,
    dataset_names=[os.path.basename(f) for f in files],
    tolerance_samples=1,
    sfreq=250.0,
    min_sequence_length=2,
    correlation_threshold=0.825
)

for cluster_id, sequences in sequences_by_cluster.items():
    print(f"\nCluster {cluster_id}: {len(sequences)} sequences")
    for sequence in sequences[:5]:
        print(f"  {sequence['dataset']}: "
              f"{sequence['start_time']:.3f}s - {sequence['end_time']:.3f}s "
              f"({sequence['length']} activations)")

stppy.io.save_pickle(sequences_by_cluster, './sequences_by_cluster.pkl')

## 6. Visualize Spatiotemporal Patterns

In [ ]:
# Cluster consensus average
for i in range(len(clusters_to_keep)):
    fig = stppy.visualization.plot_cluster_average(
            cluster_averages[clusters_to_keep[i]],
            '/home/novicky/stppy/electrode_positions.json',
            cluster_id=clusters_to_keep[i],
            vlim = [-1,1]
        )


In [ ]:
# Sample from all avalanches
fig = stppy.visualization.plot_avalanche_topomaps(
    avalanches,
    './electrode_positions.json',
    n_samples=10,
    random_seed=42
)

In [ ]:
# Sample from a specific cluster
fig = stppy.visualization.plot_avalanche_topomaps(
    avalanches,
    './electrode_positions.json',
    clusters=clusters,
    cluster_id=clusters_to_keep[0],
    n_samples=5
)

In [ ]:
# Sequence topomaps (original EEG during sequence windows)
fig = stppy.visualization.plot_sequence_topomaps(
    sequences_by_cluster,
    datasets,
    [os.path.basename(f) for f in files],
    './electrode_positions.json',
    cluster_id=clusters_to_keep[0],
    polarity='standing',
    sort_by_length=True,
    n_sequences=3
)

In [ ]:
# Video — cluster consensus pattern (requires ffmpeg)
stppy.visualization.create_cluster_video(
    cluster_averages[clusters_to_keep[0]],
    './electrode_positions.json',
    output_file='./cluster_0.mp4',
    cluster_id=clusters_to_keep[0],
    fps=5
)

In [ ]:
# Video — single avalanche (requires ffmpeg)
stppy.visualization.create_avalanche_video(
    avalanches[0],
    './electrode_positions.json',
    output_file='./avalanche_0.mp4'
)

In [ ]:
# Video — sequences from a cluster (requires ffmpeg)
stppy.visualization.create_cluster_sequences_video(
    sequences_by_cluster,
    datasets,
    [os.path.basename(f) for f in files],
    './electrode_positions.json',
    output_file='./cluster_0_sequences.mp4',
    cluster_id=clusters_to_keep[0],
    polarity='standing',
    sort_by_length=True,
    n_sequences=10,
    fps=10
)